# 📖 Notebook 4: Cache Invalidation & TTL Strategies

> *"There are only two hard things in Computer Science: cache invalidation and naming things."*  
> — Phil Karlton

The cache is only useful if the data in it is **reasonably fresh**. Stale data leads to bugs, user confusion, and broken features. This notebook covers the strategies for keeping cached data up to date.

## Learning Objectives

- Understand TTL (Time To Live) and how to choose the right duration
- Implement explicit cache invalidation on writes
- Learn eviction policies (LRU, LFU)
- Combine TTL + invalidation for robust freshness

In [ ]:
import psycopg2
import redis
import json
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "caching_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

r = redis.Redis(**REDIS_CONFIG)
r.flushdb()
print("✅ Connected and Redis cleared")

## ⏰ TTL (Time To Live)

TTL is the simplest freshness strategy: every cached value **automatically expires** after a set duration. When it expires, the next read is a cache miss, and fresh data is fetched from the database.

```
Timeline:
─────────────────────────────────────────────────────────
  0s        Cache SET with TTL=60s
  10s       Cache HIT (fresh)
  30s       Cache HIT (still fresh)
  60s       ⏰ TTL expires → key deleted automatically
  61s       Cache MISS → fetch from DB → re-cache
─────────────────────────────────────────────────────────
```

In [ ]:
def get_product_with_ttl(product_id: int, ttl_seconds: int = 10) -> dict:
    """
    Cache-aside with TTL: cached data auto-expires after ttl_seconds.
    """
    cache_key = f"product:{product_id}"
    
    cached = r.get(cache_key)
    if cached:
        remaining = r.ttl(cache_key)
        print(f"  ⚡ HIT  (TTL remaining: {remaining}s)")
        return json.loads(cached)
    
    # Cache miss — fetch from DB
    print(f"  🐌 MISS (fetching from database)")
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id, name, price FROM products WHERE id = %s", (product_id,)
    )
    row = cursor.fetchone()
    conn.close()
    
    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    
    # Store with TTL — Redis will auto-delete after ttl_seconds
    r.setex(cache_key, ttl_seconds, json.dumps(product))
    
    return product

# Demo: Watch TTL countdown
print("⏰ TTL Demo (5 second TTL):")
print()

# First call — cache miss
print("t=0s:")
product = get_product_with_ttl(42, ttl_seconds=5)

# Second call — cache hit
time.sleep(1)
print("t=1s:")
product = get_product_with_ttl(42, ttl_seconds=5)

# Third call — still hit
time.sleep(2)
print("t=3s:")
product = get_product_with_ttl(42, ttl_seconds=5)

# Wait for TTL to expire
print("\n⏳ Waiting for TTL to expire...")
time.sleep(3)

# Fourth call — cache miss again!
print("t=6s:")
product = get_product_with_ttl(42, ttl_seconds=5)

print()
print("💡 TTL ensures stale data is automatically refreshed.")
print("   The trade-off: data can be stale for up to TTL seconds.")

## 🎯 Choosing the Right TTL

There's no one-size-fits-all TTL. It depends on how fresh the data needs to be and how often it changes.

| Data Type | TTL | Why |
|-----------|-----|-----|
| User session | 30 min | Security — sessions should time out |
| Product price | 1–5 min | Prices change occasionally, small staleness OK |
| Homepage feed | 30–60 sec | Needs to feel fresh, but exact real-time not needed |
| Config/feature flags | 5–15 min | Rarely change, safe to cache longer |
| Stock quantity | 10–30 sec | Changes frequently during sales |
| Static content | 1–24 hours | Almost never changes |

In [ ]:
r.flushdb()

# Let's see how TTL affects staleness in practice

def measure_staleness(product_id: int, ttl: int, update_after: float):
    """
    Cache a product, update the DB after `update_after` seconds,
    then check how long the cache serves stale data.
    """
    # Initial cache
    product = get_product_with_ttl(product_id, ttl_seconds=ttl)
    original_price = product["price"]
    
    # Simulate a price update in the database
    time.sleep(update_after)
    conn = get_db()
    cursor = conn.cursor()
    new_price = original_price + 50
    cursor.execute("UPDATE products SET price = %s WHERE id = %s", (new_price, product_id))
    conn.commit()
    conn.close()
    
    # Check: is the cache stale?
    cached = r.get(f"product:{product_id}")
    if cached:
        cached_price = json.loads(cached)["price"]
        remaining = r.ttl(f"product:{product_id}")
        print(f"  ⚠️  Stale! Cache: ${cached_price}, DB: ${new_price} (expires in {remaining}s)")
        return remaining
    else:
        print(f"  ✅ TTL already expired — next read gets fresh data")
        return 0

print("📊 Staleness Window by TTL:")
print()

for ttl_val in [5, 10, 30]:
    print(f"TTL = {ttl_val}s (DB updated after 1s):")
    remaining = measure_staleness(ttl_val + 100, ttl=ttl_val, update_after=1)
    print(f"  Max staleness: ~{remaining}s")
    print()
    r.flushdb()

print("💡 Shorter TTL = fresher data, but more cache misses (more DB load).")
print("   Longer TTL = fewer DB hits, but data can be stale longer.")

## 🗑️ Explicit Invalidation

TTL is passive — you wait for the timer to expire. **Explicit invalidation** is active — you delete the cache entry immediately when data changes.

This is the most important technique for keeping caches fresh.

In [ ]:
r.flushdb()

def cache_product(product_id: int) -> dict:
    """Cache-aside: check cache, fall back to DB."""
    cache_key = f"product:{product_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
    row = cursor.fetchone()
    conn.close()
    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    r.setex(cache_key, 300, json.dumps(product))  # 5 min TTL as safety net
    return product

# Strategy 1: Delete on write
def update_price_invalidate(product_id: int, new_price: float):
    """Update DB, then DELETE cache key."""
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("UPDATE products SET price = %s WHERE id = %s", (new_price, product_id))
    conn.commit()
    conn.close()
    
    # Invalidate: next read will fetch fresh data from DB
    r.delete(f"product:{product_id}")

# Strategy 2: Update on write (write-through style)
def update_price_refresh(product_id: int, new_price: float):
    """Update DB, then UPDATE cache with fresh data."""
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE products SET price = %s WHERE id = %s RETURNING id, name, price",
        (new_price, product_id)
    )
    row = cursor.fetchone()
    conn.commit()
    conn.close()
    
    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    r.setex(f"product:{product_id}", 300, json.dumps(product))

# Demo: Compare both strategies
print("📋 Strategy 1: DELETE on write (invalidate)")
print("-" * 50)

product = cache_product(42)
print(f"  Cached: ${product['price']}")

update_price_invalidate(42, 199.99)
print(f"  Updated to $199.99 + deleted cache")

start = time.time()
product = cache_product(42)
miss_time = (time.time() - start) * 1000
print(f"  Next read: ${product['price']} ({miss_time:.2f}ms — cache miss, fresh from DB)")

print()
print("📋 Strategy 2: UPDATE on write (refresh)")
print("-" * 50)

update_price_refresh(42, 249.99)
print(f"  Updated to $249.99 + refreshed cache")

start = time.time()
product = cache_product(42)
hit_time = (time.time() - start) * 1000
print(f"  Next read: ${product['price']} ({hit_time:.2f}ms — cache hit, already fresh!)")

print()
print("💡 DELETE is simpler and safer (no race conditions).")
print("   UPDATE avoids the next-read miss but adds write complexity.")
print("   Most systems use DELETE + TTL as a safety net.")

## 🔗 Invalidating Related Keys

Sometimes updating one thing means multiple cache keys are stale.  
For example, updating a product price might affect:
- The product page cache
- The category listing cache
- The search results cache
- The "featured products" cache

Let's see how to handle this.

In [ ]:
r.flushdb()

# Simulate multiple caches that contain product data
def populate_related_caches(product_id: int):
    """Simulate caching product data in multiple places."""
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT p.id, p.name, p.price, c.id as cat_id, c.name as cat_name
        FROM products p JOIN categories c ON p.category_id = c.id
        WHERE p.id = %s
    """, (product_id,))
    row = cursor.fetchone()
    conn.close()
    
    product = {"id": row[0], "name": row[1], "price": float(row[2])}
    cat_id = row[3]
    
    # Cache in multiple keys
    r.setex(f"product:{product_id}", 300, json.dumps(product))
    r.setex(f"category:{cat_id}:products", 300, json.dumps([product]))  # category listing
    r.setex("featured:products", 300, json.dumps([product]))           # featured page
    
    # Track which keys are affected by this product
    # Use a Redis SET to maintain the relationship
    r.sadd(f"product:{product_id}:related_keys",
           f"product:{product_id}",
           f"category:{cat_id}:products",
           "featured:products")
    
    return product

def invalidate_product_and_related(product_id: int):
    """Delete the product cache AND all related caches."""
    related_key = f"product:{product_id}:related_keys"
    keys_to_delete = r.smembers(related_key)
    
    if keys_to_delete:
        r.delete(*keys_to_delete, related_key)
    
    return keys_to_delete

# Demo
print("🔗 Related Key Invalidation Demo:")
print()

product = populate_related_caches(42)
print(f"Cached product 42 in multiple keys:")
related = r.smembers("product:42:related_keys")
for key in sorted(related):
    print(f"  📦 {key}")

print()
deleted = invalidate_product_and_related(42)
print(f"🗑️ Invalidated {len(deleted)} related cache keys")

# Verify all are gone
remaining = [k for k in related if r.exists(k)]
print(f"   Remaining keys: {len(remaining)} (should be 0)")
print()
print("💡 Track related keys so you can invalidate everything")
print("   that might contain stale data after an update.")

## 📊 Eviction Policies: LRU vs LFU

When the cache is full, Redis needs to decide **what to evict** to make room. The two most important policies:

- **LRU (Least Recently Used)**: evict the key that hasn't been accessed for the longest time
- **LFU (Least Frequently Used)**: evict the key that has been accessed the fewest times

Let's simulate both to see the difference.

In [ ]:
class SimulatedCache:
    """A simple cache simulator to demonstrate eviction policies."""
    
    def __init__(self, max_size: int, policy: str = "lru"):
        self.max_size = max_size
        self.policy = policy
        self.cache = {}            # key → value
        self.access_order = []     # for LRU: most recent access at end
        self.access_count = {}     # for LFU: key → access count
        self.evictions = []
        self.hits = 0
        self.misses = 0
    
    def get(self, key: str):
        if key in self.cache:
            self.hits += 1
            self._record_access(key)
            return self.cache[key]
        self.misses += 1
        return None
    
    def put(self, key: str, value):
        if key not in self.cache and len(self.cache) >= self.max_size:
            self._evict()
        self.cache[key] = value
        self._record_access(key)
    
    def _record_access(self, key: str):
        if key in self.access_order:
            self.access_order.remove(key)
        self.access_order.append(key)
        self.access_count[key] = self.access_count.get(key, 0) + 1
    
    def _evict(self):
        if self.policy == "lru":
            victim = self.access_order[0]  # least recently used
        elif self.policy == "lfu":
            victim = min(self.cache.keys(), key=lambda k: self.access_count.get(k, 0))
        else:  # FIFO
            victim = self.access_order[0]
        
        del self.cache[victim]
        if victim in self.access_order:
            self.access_order.remove(victim)
        self.evictions.append(victim)

# Simulate: a cache with only 5 slots, 10 products
# Products 1-3 are "popular" (accessed often), 4-10 are "cold"

import random

access_pattern = []
for _ in range(50):
    if random.random() < 0.7:  # 70% of traffic goes to products 1-3
        access_pattern.append(f"product:{random.randint(1, 3)}")
    else:
        access_pattern.append(f"product:{random.randint(4, 10)}")

for policy in ["lru", "lfu"]:
    cache = SimulatedCache(max_size=5, policy=policy)
    
    for key in access_pattern:
        result = cache.get(key)
        if result is None:
            cache.put(key, f"data_for_{key}")
    
    total = cache.hits + cache.misses
    print(f"📊 {policy.upper()} Policy (cache size=5, 50 requests):")
    print(f"   Hits: {cache.hits}, Misses: {cache.misses}, Hit Rate: {cache.hits/total*100:.0f}%")
    print(f"   Evictions: {len(cache.evictions)}")
    print(f"   Final cache: {sorted(cache.cache.keys())}")
    print()

print("💡 LRU works well for most workloads.")
print("   LFU is better when some keys are consistently popular (trending content).")
print("   Redis uses LRU by default — it's the safe choice.")

## 🛡️ TTL + Invalidation: Belt and Suspenders

The best practice is to combine both:
- **Explicit invalidation** on every write (keeps data fresh immediately)
- **TTL as a safety net** (catches edge cases where invalidation was missed)

This way, even if your invalidation logic has a bug, the stale data will expire eventually.

In [ ]:
r.flushdb()

class ProductCache:
    """Production-style cache with TTL + explicit invalidation."""
    
    DEFAULT_TTL = 300  # 5 minutes as safety net
    
    def __init__(self):
        self.r = redis.Redis(**REDIS_CONFIG)
        self.stats = {"hits": 0, "misses": 0, "invalidations": 0}
    
    def get(self, product_id: int) -> dict:
        cached = self.r.get(f"product:{product_id}")
        if cached:
            self.stats["hits"] += 1
            return json.loads(cached)
        
        self.stats["misses"] += 1
        conn = get_db()
        cursor = conn.cursor()
        cursor.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
        row = cursor.fetchone()
        conn.close()
        
        if not row:
            return None
        
        product = {"id": row[0], "name": row[1], "price": float(row[2])}
        self.r.setex(f"product:{product_id}", self.DEFAULT_TTL, json.dumps(product))
        return product
    
    def update_price(self, product_id: int, new_price: float) -> dict:
        # Write to DB
        conn = get_db()
        cursor = conn.cursor()
        cursor.execute(
            "UPDATE products SET price = %s WHERE id = %s", (new_price, product_id)
        )
        conn.commit()
        conn.close()
        
        # Invalidate cache (next read will get fresh data)
        self.r.delete(f"product:{product_id}")
        self.stats["invalidations"] += 1
        
        return {"id": product_id, "price": new_price}
    
    def print_stats(self):
        total = self.stats["hits"] + self.stats["misses"]
        rate = self.stats["hits"] / total * 100 if total > 0 else 0
        print(f"📊 Cache Stats:")
        print(f"   Hits: {self.stats['hits']}")
        print(f"   Misses: {self.stats['misses']}")
        print(f"   Hit Rate: {rate:.1f}%")
        print(f"   Invalidations: {self.stats['invalidations']}")

# Demo: simulate a realistic workload
pc = ProductCache()

print("🔄 Simulating realistic workload:")
print("   50 reads, 3 price updates, 50 more reads")
print()

# Phase 1: warm up with reads
for _ in range(50):
    pc.get(random.randint(1, 10))

# Phase 2: some writes (with invalidation)
pc.update_price(5, 99.99)
pc.update_price(7, 149.99)
pc.update_price(3, 199.99)

# Phase 3: more reads (invalidated keys cause misses)
for _ in range(50):
    pc.get(random.randint(1, 10))

pc.print_stats()
print()
print("💡 Invalidation + TTL = best of both worlds.")
print("   Invalidation keeps data fresh. TTL catches what invalidation misses.")

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Redis cleared")

## 📚 Summary

### Invalidation Strategies

| Strategy | When to Use | Pros | Cons |
|----------|------------|------|------|
| **TTL only** | Data changes rarely | Simple | Stale for up to TTL |
| **Delete on write** | Most cases | Fresh immediately | Need to track all writes |
| **Update on write** | High-traffic keys | No miss after update | Write is more complex |
| **TTL + Delete** | Production systems | Robust | Slightly more code |

### Key Takeaways

1. **TTL is a safety net**, not a primary strategy — always pair it with explicit invalidation
2. **Delete on write** is simpler and safer than update on write
3. **Track related keys** when one update affects multiple caches
4. **LRU is the default** eviction policy — use it unless you have a reason not to
5. **Short TTL for volatile data**, long TTL for stable data

### Next Up

In **Notebook 5**, we'll tackle the scariest caching problems: **cache stampede** (thundering herd) and **hot keys** — and how to prevent them.